# 🚀 SC-Mamba: Training & Per-Dataset Evaluation Pipeline

**Notebook Purpose:** Pre-train SC-Mamba on synthetic priors, then run per-dataset zero-shot evaluation on any subset of the 17 GluonTS benchmark datasets.

| Step | Description | Output |
|------|-------------|--------|
| **0** | Environment Setup (CUDA 12.1) | Dependencies installed |
| **1** | Dataset Selection (interactive) | `selected` list |
| **2** | Training Config | `core/config.batch_ddp.yaml` |
| **3** | Pre-training on Synthetic Data | `models/<prefix>.pth` checkpoint |
| **4** | Per-Dataset Zero-Shot Evaluation | MASE, sMAPE, NRMSE, CRPS |
| **5** | Results Visualization | `results/<model>_benchmark_summary.png` |

> **Hardware:** Minimum NVIDIA T4 GPU (Colab). For Traffic + Weather full eval → use A100.
> **Quick Validation Mode:** Set `QUICK_VALIDATE = True` to run 6 fast datasets (~10 min on T4).

## Step 0: Environment Setup (PyTorch 2.4 + CUDA 12.1)

Installs pre-compiled `causal-conv1d` and `mamba-ssm` wheels — avoids 45+ min ninja compilation.

In [ ]:
!pip uninstall -y torch torchvision torchaudio transformers sentence-transformers 2>/dev/null
!pip install torch==2.4.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q
!pip install transformers==4.39.3 packaging triton==3.0.0 -q

# Pre-compiled CUDA wheels (avoids 45+ min ninja compile)
!wget -qO causal_conv1d.whl 'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.4.0/causal_conv1d-1.4.0%2Bcu122torch2.4cxx11abiFALSE-cp312-cp312-linux_x86_64.whl'
!wget -qO mamba_ssm.whl 'https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4%2Bcu12torch2.4cxx11abiFALSE-cp312-cp312-linux_x86_64.whl'
!pip install causal_conv1d.whl mamba_ssm.whl -q

!pip install gluonts yfinance tqdm utilsforecast pyyaml pandas numpy matplotlib seaborn -q
print('✅ Environment ready.')

## Step 1: Dataset Selection

Choose which datasets to evaluate after training. Set `SELECTED_DATASETS = 'all'` for full 17-dataset suite, or a list of GluonTS keys for a subset.

## ⚡ Speed-Up Options (Colab T4 Tips)

With **default config** (`training_rounds=1000, num_epochs=50, max_seq_len=512`):  
→ ~**10 min/epoch** × 50 epochs = **~8 hrs total** on T4.

**Why NLL goes negative?** Normal — Gaussian NLL = `0.5·log(2π·σ²) + 0.5·(y-μ)²/σ²`.  
When σ² is small (confident model), the log term goes negative. Not a bug.

**KL growing slowly?** Also correct — beta-annealing ramps KL weight from 0→0.1 over first half of training, preventing KL collapse.

| Goal | Changes | New epoch time | Total time |
|------|---------|----------------|------------|
| 🚀 Quick proof-of-concept | `training_rounds=200, num_epochs=5` | ~2 min | ~10 min |
| ⚖️ Balanced (decent quality) | `training_rounds=500, num_epochs=20` | ~5 min | ~1.5 hrs |
| 📊 Paper-grade (full) | `training_rounds=1000, num_epochs=50` | ~10 min | ~8 hrs |
| 🏎️ A100 full run | `training_rounds=1000, num_epochs=50` | ~2.5 min | ~2 hrs |

**To speed up, change these keys in Step 2:**
```python
'training_rounds': 200,   # batches per epoch
'num_epochs'     : 5,     # total epochs
'max_seq_len'    : 128,   # shorter context → ~3× faster per batch
```

> 💡 **Resume**: Set `continue_training: True` + `model_prefix` to your Drive path to resume from last checkpoint.

In [ ]:
# =====================================================================
# DATASET REGISTRY — All 17 Mamba4Cast benchmark datasets
# Columns: (gluonts_key, pred_len, freq, display_name, domain)
# Source: Table 3 of Mamba4Cast (arxiv:2410.09385)
# =====================================================================
DATASET_CONFIGS = [
    ('car_parts_without_missing',      12, 'M', 'Car Parts',        'Retail'),
    ('cif_2016',                       12, 'M', 'CIF 2016',         'Finance/Competition'),
    ('covid_deaths',                   30, 'D', 'Covid Deaths',     'Epidemiology'),
    ('ercot',                          24, 'H', 'ERCOT Load',       'Energy'),
    ('exchange_rate',                  30, 'B', 'Exchange Rate',    'Forex Finance'),
    ('fred_md',                        12, 'M', 'FRED-MD',          'Macroeconomics'),
    ('hospital',                       12, 'M', 'Hospital',         'Healthcare'),
    ('m1_monthly',                     18, 'M', 'M1 Monthly',       'Economics'),
    ('m1_quarterly',                    8, 'Q', 'M1 Quarterly',     'Economics'),
    ('m3_monthly',                     18, 'M', 'M3 Monthly',       'Economics'),
    ('m3_quarterly',                    8, 'Q', 'M3 Quarterly',     'Economics'),
    ('nn5_daily_without_missing',      56, 'D', 'NN5 Daily',        'Banking'),
    ('nn5_weekly',                      8, 'W', 'NN5 Weekly',       'Banking'),
    ('tourism_monthly',                24, 'M', 'Tourism Monthly',  'Tourism'),
    ('tourism_quarterly',               8, 'Q', 'Tourism Quarterly','Tourism'),
    ('traffic',                        24, 'H', 'Traffic',          'Transportation'),
    ('weather',                        30, 'D', 'Weather',          'Meteorology'),
]

# =====================================================================
# ⚙️  USER CONFIGURATION — Edit below
# =====================================================================
# SELECTED_DATASETS options:
#   'all'                        → all 17 datasets (~60-90 min on T4)
#   ['cif_2016', 'm1_monthly']   → specific subset
# Quick validation (~10 min): set QUICK_VALIDATE = True
SELECTED_DATASETS = 'all'
MODEL_NAME        = 'SCMamba_v1'    # MUST match generate_model_save_name(config) = f'SCMamba_{version}'
QUICK_VALIDATE    = False            # True = 6 fast datasets only

QUICK_KEYS = ['cif_2016', 'm1_quarterly', 'nn5_weekly', 'tourism_quarterly', 'm1_monthly', 'hospital']

if QUICK_VALIDATE:
    selected = [d for d in DATASET_CONFIGS if d[0] in QUICK_KEYS]
elif SELECTED_DATASETS == 'all':
    selected = DATASET_CONFIGS
else:
    selected = [d for d in DATASET_CONFIGS if d[0] in SELECTED_DATASETS]

print(f'📋 Selected {len(selected)} datasets:')
for key, pred, freq, name, domain in selected:
    print(f'   [{domain:20s}] {name:25s} freq={freq}, pred_len={pred}')


## Step 2: Training Configuration

Generates `core/config.batch_ddp.yaml`. Adjust hyperparameters here. Key SC-Mamba differences vs Mamba4Cast:
- `loss: nll` (NLL+KL ELBO instead of MSE)
- `beta_kl: 0.1` (balances spectral ELBO)
- `pred_len_sample: True` (random pred length per batch → better generalization)

In [ ]:
import yaml, os

config = {
    'seed': 42,
    'debugging': False,
    'scaler': 'min_max',
    'sin_pos_enc': False,
    'sin_pos_const': False,
    'sub_day': False,
    'encoding_dropout': 0.1,
    'handle_constants_model': True,
    'num_assets': 1,
    'ssm_config': {
        'bidirectional': False,
        'enc_conv': True,
        'init_dil_conv': True,
        'enc_conv_kernel': 5,
        'init_conv_kernel': 5,
        'init_conv_max_dilation': 3,
        'global_residual': False,
        'in_proj_norm': False,
        'initial_gelu_flag': True,
        'linear_seq': 15,
        'mamba2': False,
        'norm': True,
        'norm_type': 'layernorm',
        'num_encoder_layers': 2,
        'd_state': 128,
        'residual': False,
        'token_embed_len': 1024,
    },
    # ---- Optimization (Mamba4Cast-compatible recipe) ----
    'lr_scheduler': 'cosine',
    'initial_lr': 1e-4,
    'learning_rate': 1e-6,
    't_max': -1,
    'num_epochs': 50,
    'training_rounds': 1000,
    'validation_rounds': 50,
    'real_test_interval': 10,
    'real_test_datasets': [d[0] for d in selected[:2]],
    # ---- SC-Mamba-specific loss configuration ----
    'loss': 'nll',        # NLL + KL ELBO (vs MSE in Mamba4Cast)
    'beta_kl': 0.1,       # weight of spectral KL divergence term
    'multipoint': False,
    'sample_multi_pred': 0.0,
    # ---- Checkpoint ----
    'wandb': False,
    'continue_training': False,
    'model_prefix': 'models',
    'version': 'v1',
    # ---- Sequence sampling (Mamba4Cast: [30, 512] context, [10, 60] pred) ----
    'min_seq_len': 30,
    'max_seq_len': 512,
    'pred_len': 24,
    'pred_len_min': 10,
    'pred_len_sample': True,   # random pred len → better zero-shot generalization
    'batch_size': 64,
    'no_pos_enc': True,
    # ---- Synthetic Prior configuration (70% GP + 30% ForecastPFN) ----
    'prior_config': {
        'prior_mix_frac': 0.7,
        'curriculum_learning': False,
        'gp_prior_config': {
            'use_original_gp': False,
            'kernel_bank': {
                'matern_kernel': 0.2,
                'linear_kernel': 0.2,
                'periodic_kernel': 0.2,
                'polynomial_kernel': 0.2,
                'spectral_mixture_kernel': 0.2,
            },
            'max_kernels': 3,
            'likelihood_noise_level': 0.01,
            'noise_level': 0.05,
            'peak_spike_ratio': 0.1,
            'gaussians_periodic': True,
            'max_period_ratio': 0.5,
            'subfreq_ratio': 0.1,
            'periods_per_freq': 2,
            'gaussian_sampling_ratio': 0.5,
        },
        'fp_options': {
            'transition_ratio': 0.1,
            'freq_features': True,
            'trend_exp': False,
            'scale_noise': [0.1, 0.5],
            'seasonal_only': False,
        },
        'mixup_prob': 0.1,
        'mixup_series': 2,
        'damp_and_spike': True,
        'damping_noise_ratio': 0.1,
        'spike_noise_ratio': 0.1,
        'spike_signal_ratio': 0.05,
        'spike_batch_ratio': 0.05,
    },
}

os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

config_path = 'core/config.batch_ddp.yaml'
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f'✅ Config written → {config_path}')
print(f'   loss={config["loss"]} | beta_kl={config["beta_kl"]} | batch={config["batch_size"]}')
print(f'   seq_len=[{config["min_seq_len"]}, {config["max_seq_len"]}] | epochs={config["num_epochs"]}')
print(f'   Live monitoring on: {config["real_test_datasets"]}')

## Step 2.5: Pre-Flight Data Check & Resume Setup

**Verifies** that real validation datasets are downloaded (`.pkl` files).  
Required for the `real_test_interval` validation step during training.  
If missing, auto-generates them from GluonTS.

Also sets up **resume from checkpoint** if previous training was interrupted.

In [ ]:
import os, subprocess

# =====================================================================
# 1. Verify real validation datasets exist (pkl files)
# =====================================================================
REAL_VAL_DIR = 'data/real_val_datasets'
os.makedirs(REAL_VAL_DIR, exist_ok=True)

# Check which datasets are needed for real_test_datasets
real_test_ds = config.get('real_test_datasets', [])
padded = 'pad' if False else 'nopad'  # matches real_data_args.yaml pad=false
MAX_LEN = 512

missing = []
for ds in real_test_ds:
    pkl_name = f'{ds}_{padded}_{MAX_LEN}.pkl'
    pkl_path = os.path.join(REAL_VAL_DIR, pkl_name)
    if os.path.exists(pkl_path):
        size_mb = os.path.getsize(pkl_path) / (1024*1024)
        print(f'  ✅ {ds:30s} → {pkl_name} ({size_mb:.1f} MB)')
    else:
        missing.append(ds)
        print(f'  ❌ {ds:30s} → MISSING')

if missing:
    print(f'\n🔄 Generating {len(missing)} missing dataset(s) from GluonTS...')
    print('   This downloads from AWS and converts to pkl. May take 1-5 min per dataset.')
    !python data/scripts/store_real_datasets.py
    # Verify again
    for ds in missing:
        pkl_path = os.path.join(REAL_VAL_DIR, f'{ds}_{padded}_{MAX_LEN}.pkl')
        if os.path.exists(pkl_path):
            print(f'  ✅ {ds} → OK')
        else:
            print(f'  ⚠️  {ds} → STILL MISSING. Check store_real_datasets.py output above.')
else:
    print(f'\n✅ All {len(real_test_ds)} real validation datasets found.')

# =====================================================================
# 2. Resume from checkpoint (if training was interrupted)
# =====================================================================
# Set continue_training=True to resume from the last saved checkpoint.
# The training loop will load optimizer state, scheduler state, and epoch.
RESUME_TRAINING = False    # <---- SET TO True TO RESUME

if RESUME_TRAINING:
    config['continue_training'] = True
    with open(config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)
    print('🔄 Resume mode enabled: will load from last checkpoint.')
else:
    print('ℹ️  Fresh training. Set RESUME_TRAINING=True to resume from checkpoint.')

## Step 3: Pre-Training on Synthetic Data

SC-Mamba learns universal temporal patterns from GP + ForecastPFN synthetic priors before seeing real data.  
Checkpoint saved to `models/sc_mamba_v1.pth`.

> ⏱ **Estimated times:**
> - Full training (1000 rounds × 50 epochs): **~6–10 hrs on A100**
> - Quick prototype (100 rounds × 5 epochs): **~30–60 min on T4**
>
> **To reduce time:** Edit `training_rounds` and `num_epochs` in Step 2, or use `continue_training: True` to resume from checkpoint.

In [ ]:
# Mount Google Drive first if running on Colab to persist checkpoint:
# from google.colab import drive; drive.mount('/content/drive')
# Then set model_prefix in config to '/content/drive/MyDrive/sc_mamba_checkpoints'

!python core/train.py -c core/config.batch_ddp.yaml

In [ ]:
# --- Verify checkpoint saved ---
!ls -lh models/

## Step 4: Per-Dataset Zero-Shot Evaluation

Evaluates the trained model on each selected dataset.  
**Metrics computed:**
- **MASE** — primary metric for Table 3 comparison with Mamba4Cast
- **sMAPE** — supplementary scale-independent metric
- **NRMSE** — normalized RMSE for absolute error characterization  
- **CRPS** — probabilistic quality metric (SC-Mamba unique advantage)
- **NLL** — negative log-likelihood of the Normal(μ, σ²) output distribution

Results are saved to `results/<MODEL_NAME>_<dataset>.json` for caching.

In [ ]:
import subprocess, yaml, os
import pandas as pd
from pathlib import Path

# =====================================================================
# eval_real_dataset.py interface:
#   python eval_real_dataset.py -m <model_name>
#   → loops ALL 17 datasets internally
#   → caches per-dataset results at:
#     data/real_data_evals/<model_name>/multipoint/<dataset>_512.yml
#
# MODEL_NAME must match generate_model_save_name(config) = 'SCMamba_v1'
# =====================================================================

EVAL_MODEL_NAME = MODEL_NAME  # from Step 1 cell, should be 'SCMamba_v1'
PRED_STYLE = 'multipoint'
EVAL_DIR = Path(f'data/real_data_evals/{EVAL_MODEL_NAME}/{PRED_STYLE}')
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# --- Check which selected datasets already have cached results ---
cached, missing = [], []
for gluonts_key, pred_len, freq, name, domain in selected:
    yml_path = EVAL_DIR / f'{gluonts_key}_512.yml'
    if yml_path.exists():
        cached.append(name)
    else:
        missing.append(name)

print(f'Model: {EVAL_MODEL_NAME}')
print(f'Eval dir: {EVAL_DIR}')
print(f'✅ Cached: {len(cached)} | ❌ Missing: {len(missing)}')
if missing:
    print(f'   Will evaluate: {missing}')

# --- Run eval_real_dataset.py (skips datasets with existing .yml) ---
if missing:
    cmd = ['python', 'core/eval_real_dataset.py', '-m', EVAL_MODEL_NAME]
    print(f'\n▶ Running: {" ".join(cmd)}')
    result = subprocess.run(cmd, capture_output=False, text=True)
    if result.returncode != 0:
        print(f'⚠️  eval_real_dataset.py exited with code {result.returncode}')
else:
    print('\n✅ All selected datasets already cached. Skipping eval.')

# --- Load results from YAML caches ---
results = []
for gluonts_key, pred_len, freq, name, domain in selected:
    yml_path = EVAL_DIR / f'{gluonts_key}_512.yml'
    if not yml_path.exists():
        results.append({'dataset': name, 'key': gluonts_key, 'domain': domain,
                        'freq': freq, 'pred_len': pred_len, 'status': 'MISSING'})
        continue
    with open(yml_path) as f:
        raw = yaml.safe_load(f)
    if raw is None:
        results.append({'dataset': name, 'key': gluonts_key, 'domain': domain,
                        'freq': freq, 'pred_len': pred_len, 'status': 'EMPTY'})
        continue
    # raw format: {'512_<pred_len>': {'mase': ..., 'mae': ..., ...}}
    metrics = next(iter(raw.values()), {})
    if metrics.get('mase') is None:
        results.append({'dataset': name, 'key': gluonts_key, 'domain': domain,
                        'freq': freq, 'pred_len': pred_len, 'status': 'PLACEHOLDER'})
        continue
    results.append({
        'dataset': name, 'key': gluonts_key, 'domain': domain,
        'freq': freq, 'pred_len': pred_len, 'status': 'OK',
        'MASE': metrics.get('mase'), 'MAE': metrics.get('mae'),
        'RMSE': metrics.get('rmse'), 'sMAPE': metrics.get('smape'),
        'NLL': metrics.get('nll'), 'CRPS': metrics.get('crps'),
    })
    print(f'✅ [{domain:20s}] {name:25s} MASE={metrics.get("mase","?"):.4f}  CRPS={metrics.get("crps","?"):.4f}')

df = pd.DataFrame(results)
csv_path = f'results/{EVAL_MODEL_NAME}_all_results.csv'
os.makedirs('results', exist_ok=True)
df.to_csv(csv_path, index=False)

print(f'\n{"="*60}')
print(f'📊 SUMMARY — Model: {EVAL_MODEL_NAME}')
print(f'{"="*60}')
ok = df[df['status'] == 'OK']
if not ok.empty:
    cols = ['dataset', 'domain', 'freq', 'pred_len']
    for m in ['MASE','sMAPE','CRPS','NLL']:
        if m in ok.columns: cols.append(m)
    display(ok[cols].round(4))
    print(f'\n   Avg MASE: {ok["MASE"].mean():.4f}')
    if 'CRPS' in ok.columns:
        print(f'   Avg CRPS: {ok["CRPS"].mean():.4f}')
err = df[df['status'] != 'OK']
if not err.empty:
    print(f'\n⚠️  {len(err)} datasets not evaluated:')
    for _, r in err.iterrows():
        print(f'   [{r["status"]:12s}] {r["dataset"]}')
print(f'\n💾 Saved → {csv_path}')

## Step 5: Visualization

Performance heatmap across datasets with domain coloring and median reference line.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd, os

results_path = f'results/{MODEL_NAME}_all_results.csv'
if not os.path.exists(results_path):
    print('⚠️  Run Step 4 first.')
else:
    df = pd.read_csv(results_path)
    df_ok = df[df['status'] == 'OK'].copy()
    
    metric_cols = [m for m in ['MASE', 'sMAPE', 'CRPS'] if m in df_ok.columns]
    fig, axes = plt.subplots(1, len(metric_cols), figsize=(7*len(metric_cols), 7))
    if len(metric_cols) == 1: axes = [axes]
    
    fig.suptitle(f'SC-Mamba ({MODEL_NAME}) — Zero-Shot Benchmark', fontsize=14, fontweight='bold')
    
    for ax, metric in zip(axes, metric_cols):
        data = df_ok.set_index('dataset')[metric].sort_values()
        med = data.median()
        colors = ['#43A047' if v <= med else '#EF5350' for v in data]
        bars = ax.barh(data.index, data.values, color=colors, edgecolor='white')
        ax.axvline(med, color='#212121', linestyle='--', alpha=0.6, linewidth=1.2, label=f'Median={med:.3f}')
        ax.set_xlabel(metric, fontsize=11)
        ax.set_title(f'{metric} by Dataset (↓ better)', fontsize=12)
        ax.legend(fontsize=9)
        for bar, val in zip(bars, data.values):
            ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                   f'{val:.3f}', va='center', fontsize=8)
    
    plt.tight_layout()
    out_path = f'results/{MODEL_NAME}_benchmark_summary.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Saved → {out_path}')